# 第 7 章代码实验：异常与资源管理

每题先读代码并填写预测，再运行该题，立即运行本题检查，最后记录异常走过哪些调用层、哪些语句被跳过、资源是否释放。把整个 `"__未填写__"`（包括引号）替换为实际 Python 值。两位编号只用于匹配字段，中文描述说明观察对象，不提示结果类型。

实验 16～18 是实现题，19～22 是深入观察题。Notebook 不提供实现答案；检查器只核对客观行为，不评价你对资源所有权与处理策略的解释。

## 实验 1：语法检查与运行时转换

先区分被检查的源码和当前正在运行的有效代码，再预测两次失败各在什么阶段发生。

In [ ]:
syntax_name = None
try:
    compile("def broken(:\n    pass", "<study>", "exec")
except Exception as error:
    syntax_name = type(error).__name__

runtime_name = None
try:
    int("bad")
except Exception as error:
    runtime_name = type(error).__name__

print(syntax_name, runtime_name)

observation_01 = {
    "01. compile(...) 记录的异常名称": "__未填写__",
    "02. int('bad') 记录的异常名称": "__未填写__",
}

### 实验 1 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 1)

### 实验 1 原因解释

为什么外层代码能够捕获被检查字符串的语法问题，而不能在同一份无效源码里先运行自己的 `try`？

## 实验 2：异常类型的继承关系

按类继承关系预测四个表达式。不要根据异常名称猜测父子关系。

In [ ]:
results = (
    issubclass(KeyError, LookupError),
    isinstance(KeyboardInterrupt(), Exception),
    isinstance(KeyboardInterrupt(), BaseException),
    isinstance(ValueError("bad"), Exception),
)
print(results)

observation_02 = {
    "01. issubclass(KeyError, LookupError) 的结果": "__未填写__",
    "02. KeyboardInterrupt 实例是否属于 Exception": "__未填写__",
    "03. KeyboardInterrupt 实例是否属于 BaseException": "__未填写__",
    "04. ValueError 实例是否属于 Exception": "__未填写__",
}

### 实验 2 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 2)

### 实验 2 原因解释

为什么 `except Exception` 与裸 `except:` 捕获范围不同？

## 实验 3：异常逐层向外传播

写出完整事件顺序；标出内层抛出后哪一句普通代码没有执行。

In [ ]:
events = []

def inner():
    events.append("inner")
    raise ValueError("bad")

def middle():
    events.append("middle before")
    inner()
    events.append("middle after")

try:
    middle()
except ValueError:
    events.append("outer caught")
print(events)

observation_03 = {
    "01. 调用结束后 events 的完整顺序": "__未填写__",
}

### 实验 3 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 3)

### 实验 3 原因解释

异常先经过哪一层，再由哪一层处理？`middle after` 为什么没有加入？

## 实验 4：具体异常与父类分支的顺序

比较两个函数里 `except` 的排列，预测各自返回的标记。

In [ ]:
def specific_first():
    try:
        {}["missing"]
    except KeyError:
        return "specific"
    except LookupError:
        return "general"

def general_first():
    try:
        {}["missing"]
    except LookupError:
        return "general"
    except KeyError:
        return "specific"

print(specific_first(), general_first())

observation_04 = {
    "01. specific_first() 的结果": "__未填写__",
    "02. general_first() 的结果": "__未填写__",
}

### 实验 4 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 4)

### 实验 4 原因解释

为什么第二个函数中更具体的分支没有机会执行？

## 实验 5：正常和失败路径上的 `else` 与 `finally`

分别写出两条路径完整的事件列表，而不是只说哪几个子句会执行。

In [ ]:
def flow(fail):
    events = []
    try:
        events.append("try")
        if fail:
            raise ValueError("failed")
    except ValueError:
        events.append("except")
    else:
        events.append("else")
    finally:
        events.append("finally")
    return events

success = flow(False)
failure = flow(True)
print(success, failure)

observation_05 = {
    "01. flow(False) 的事件顺序": "__未填写__",
    "02. flow(True) 的事件顺序": "__未填写__",
}

### 实验 5 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 5)

### 实验 5 原因解释

`else` 需要 `try` 以什么方式结束？`finally` 在两条路径各位于哪里？

## 实验 6：`try` 中提前返回

预测返回值和事件列表；重点判断 `else` 与 `finally` 是否运行。

In [ ]:
events = []

def early_return():
    try:
        events.append("try")
        return "body"
    except ValueError:
        events.append("except")
    else:
        events.append("else")
    finally:
        events.append("finally")

result = early_return()
print(result, events)

observation_06 = {
    "01. early_return() 的结果": "__未填写__",
    "02. 调用结束后 events 的内容": "__未填写__",
}

### 实验 6 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 6)

### 实验 6 原因解释

为什么函数准备返回时仍经过 `finally`，却没有运行 `else`？

## 实验 7：`finally` 中的新返回值

比较原路径准备抛异常与准备返回时，`finally` 中的 `return` 会产生什么影响。

In [ ]:
def masked():
    try:
        raise ValueError("important")
    finally:
        return "hidden"

def overridden():
    try:
        return "old"
    finally:
        return "new"

print(masked(), overridden())

observation_07 = {
    "01. masked() 的调用结果": "__未填写__",
    "02. overridden() 的调用结果": "__未填写__",
}

### 实验 7 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 7)

### 实验 7 原因解释

如果清理代码里必须返回一个状态，会失去原来的哪种信息？

## 实验 8：捕获后重新抛出

写出内外两层处理顺序，说明裸 `raise` 让谁继续收到原异常。

In [ ]:
events = []

def convert(text):
    try:
        return int(text)
    except ValueError:
        events.append("inner caught")
        raise

try:
    convert("bad")
except ValueError:
    events.append("outer caught")
print(events)

observation_08 = {
    "01. 两层处理后的 events": "__未填写__",
}

### 实验 8 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 8)

### 实验 8 原因解释

这里的内层有没有把异常变成合法返回值？裸 `raise` 与新建一个异常对象有何区别？

## 实验 9：显式异常链中的对象关系

预测新异常的原因、上下文是否指向同一个原异常对象；不要只看错误消息。

In [ ]:
original = None
try:
    try:
        int("bad")
    except ValueError as caught:
        original = caught
        raise RuntimeError("conversion failed") from caught
except RuntimeError as error:
    cause_is_original = error.__cause__ is original
    context_is_original = error.__context__ is original
    suppress_context = error.__suppress_context__

print(cause_is_original, context_is_original, suppress_context)

observation_09 = {
    "01. error.__cause__ is original 的结果": "__未填写__",
    "02. error.__context__ is original 的结果": "__未填写__",
    "03. error.__suppress_context__ 的结果": "__未填写__",
}

### 实验 9 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 9)

### 实验 9 原因解释

`from caught` 对调用方查看根因有什么帮助？`from None` 会真的删除原对象吗？

## 实验 10：自定义异常保存结构化信息

观察调用方怎样根据异常类型读取字段，而不解析错误消息。

In [ ]:
class InvalidFieldError(Exception):
    def __init__(self, field, message):
        self.field = field
        super().__init__(message)

try:
    raise InvalidFieldError("age", "must be non-negative")
except InvalidFieldError as error:
    field = error.field
    message = str(error)
print(field, message)

observation_10 = {
    "01. 捕获后 error.field 的值": "__未填写__",
    "02. 捕获后 str(error) 的值": "__未填写__",
}

### 实验 10 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 10)

### 实验 10 原因解释

什么信息适合单独保存为属性？为什么不让调用方从消息中截取字段名？

## 实验 11：读取缺失字典键与默认值处理

读取不存在的键，观察默认处理是否修改原字典。

In [ ]:
settings = {"timeout": 5}
try:
    default_value = settings["missing"]
except KeyError:
    default_value = 10
print(default_value, settings)

observation_11 = {
    "01. 失败读取后 default_value 的值": "__未填写__",
    "02. 失败读取后 settings 的完整内容": "__未填写__",
}

### 实验 11 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 11)

### 实验 11 原因解释

若把整个业务流程放进同一个 `try`，另一处 `KeyError` 会被误认为哪一步失败？

## 实验 12：`finally` 清理内存资源

故意在使用资源期间失败，记录清理与外层捕获的先后，以及最终关闭状态。

In [ ]:
from io import StringIO

buffer = StringIO("abc")
events = []
try:
    try:
        events.append("use")
        raise ValueError("failed")
    finally:
        buffer.close()
        events.append("close")
except ValueError:
    events.append("outer caught")
print(events, buffer.closed)

observation_12 = {
    "01. 资源失败路径中的 events 顺序": "__未填写__",
    "02. 最后 buffer.closed 的值": "__未填写__",
}

### 实验 12 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 12)

### 实验 12 原因解释

为什么清理发生在外层捕获之前？若把 `close()` 放在 `try` 之后会怎样？

## 实验 13：类式上下文管理器的正常路径

预测 `__enter__()`、代码块和 `__exit__()` 的顺序，并判断 `as` 绑定了什么。

In [ ]:
events = []

class Scope:
    def __enter__(self):
        events.append("enter")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        events.append("exit:" + (exc_type.__name__ if exc_type else "none"))
        return False

manager = Scope()
with manager as active:
    same_object = active is manager
    events.append("body")
print(events, same_object)

observation_13 = {
    "01. 正常离开后 events 的顺序": "__未填写__",
    "02. active is manager 的结果": "__未填写__",
}

### 实验 13 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 13)

### 实验 13 原因解释

正常路径传给 `__exit__()` 的异常相关参数是什么？

## 实验 14：上下文管理器抑制与传播异常

同一管理器只改变 `__exit__()` 返回值，比较外层是否还能接到异常。

In [ ]:
class Switch:
    def __init__(self, events, suppress):
        self.events = events
        self.suppress = suppress

    def __enter__(self):
        self.events.append("enter")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.events.append("exit:" + (exc_type.__name__ if exc_type else "none"))
        return self.suppress

propagated = []
try:
    with Switch(propagated, False):
        propagated.append("body")
        raise ValueError("first")
except ValueError:
    propagated.append("outer caught")

suppressed = []
with Switch(suppressed, True):
    suppressed.append("body")
    raise ValueError("second")
suppressed.append("continued")
print(propagated, suppressed)

observation_14 = {
    "01. 不抑制时的完整事件顺序": "__未填写__",
    "02. 抑制时的完整事件顺序": "__未填写__",
}

### 实验 14 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 14)

### 实验 14 原因解释

返回真值是在处理异常，还是只改变了传播？为什么业务管理器不能无条件这么做？

## 实验 15：生成器式上下文管理器的两条路径

跟踪 `yield` 前后的事件；异常路径是否仍执行清理并到达外层？

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed(events):
    events.append("acquire")
    try:
        yield "resource"
    finally:
        events.append("release")

success = []
with managed(success) as resource:
    success.append("body")

failure = []
try:
    with managed(failure):
        failure.append("body")
        raise ValueError("failed")
except ValueError:
    failure.append("outer caught")
print(success, failure)

observation_15 = {
    "01. 正常路径的事件顺序": "__未填写__",
    "02. 失败路径的事件顺序": "__未填写__",
}

### 实验 15 即时检查

先运行上面的实验单元格，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 15)

### 实验 15 原因解释

`@contextmanager` 在异常路径把失败送回生成器的哪个位置？清理后为何仍能被外层捕获？

## 实验 16：实现正整数转换接口

保留给出的 `InputNumberError` 类，实现 `parse_positive_int(text)`。输入 `text` 为字符串：`int(text)` 能转换且结果大于 `0` 时返回整数；转换失败时抛出 `InputNumberError`，并通过显式异常链保留原 `ValueError`；转换成功但结果非正时抛出 `InputNumberError`，不伪造转换原因。示例：`" 7 "` 返回 `7`；`"0"` 与 `"bad"` 都不返回合法整数，但两种失败的原因不同。

In [ ]:
class InputNumberError(Exception):
    pass


def parse_positive_int(text):
    raise NotImplementedError("请实现 parse_positive_int")

### 实验 16 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 16)

### 实验 16 原因解释

把“无法转换”和“已转换但不在允许范围”分成两条路径；哪一条有原始转换异常？

## 实验 17：实现事务式上下文管理器

实现 `Transaction(events)` 类。进入 `with` 时向给定列表追加 `"begin"` 并把管理器自身交给 `as`；正常退出追加 `"commit"`；代码块抛出异常时追加 `"rollback"`，且原异常继续向外传播。不要在管理器内假装执行真实数据库操作。

In [ ]:
class Transaction:
    def __init__(self, events):
        raise NotImplementedError("请初始化 Transaction")

    def __enter__(self):
        raise NotImplementedError("请实现进入逻辑")

    def __exit__(self, exc_type, exc_value, traceback):
        raise NotImplementedError("请实现退出逻辑")

### 实验 17 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 17)

### 实验 17 原因解释

`__exit__()` 怎样知道本次是正常结束还是失败？为什么失败后仍要让调用方看见原异常？

## 实验 18：实现生成器式缓冲区管理器

使用标准库 `@contextmanager` 实现 `open_buffer(text)`，进入后交出一个可读取 `text` 的 `io.StringIO` 对象；正常或异常离开时都关闭它，并让代码块中的异常继续传播。要求生成器式管理器，不能只返回一个没有生命周期控制的缓冲区对象。

In [ ]:
from contextlib import contextmanager
from io import StringIO


@contextmanager
def open_buffer(text):
    raise NotImplementedError("请实现 open_buffer")

### 实验 18 即时检查

先运行定义单元格，再检查当前实现。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 18)

### 实验 18 原因解释

资源在哪个时刻创建？为什么清理代码需要覆盖代码块抛出异常的路径？

## 实验 19：两种重新抛出的 traceback

比较裸 `raise` 与 `raise error` 后的相关函数帧名称。只看函数名，不依赖本机文件路径和行号。

In [ ]:
import traceback

def fail():
    raise ValueError("bad")

def reraised_bare():
    try:
        fail()
    except ValueError:
        raise

def reraised_named():
    try:
        fail()
    except ValueError as error:
        raise error

def relevant_names(function):
    try:
        function()
    except ValueError as error:
        frames = traceback.extract_tb(error.__traceback__)
        return [frame.name for frame in frames if frame.name in ("reraised_bare", "reraised_named", "fail")]

bare_names = relevant_names(reraised_bare)
named_names = relevant_names(reraised_named)
print(bare_names, named_names)

observation_19 = {
    "01. 裸 raise 路径的相关函数名顺序": "__未填写__",
    "02. raise error 路径的相关函数名顺序": "__未填写__",
}

### 实验 19 即时检查

先运行本题代码，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 19)

### 实验 19 原因解释

两种路径都还保留哪个最早的失败函数？哪一种额外记录了重新抛出的位置？

## 实验 20：检查后状态发生变化

先记录检查结果，再模拟其他操作修改字典，观察实际读取的结果。

In [ ]:
state = {"timeout": 5}
was_present = "timeout" in state
state.pop("timeout")
try:
    value_after_change = state["timeout"]
except KeyError:
    value_after_change = "missing at use"
print(was_present, value_after_change, state)

observation_20 = {
    "01. 检查时 was_present 的值": "__未填写__",
    "02. 实际读取后 value_after_change 的值": "__未填写__",
    "03. 读取后的 state 内容": "__未填写__",
}

### 实验 20 即时检查

先运行本题代码，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 20)

### 实验 20 原因解释

为什么一次先前的检查不能保证之后的读取？把实际读取放在窄范围的 `try` 里解决了什么，又没有解决什么？

## 实验 21：部分资源取得失败

取得第一个资源后，第二次取得失败。预测清理、外层捕获的事件顺序和第一个资源的最终状态。

In [ ]:
from io import StringIO

events = []
handles = []

def acquire_pair():
    first = StringIO("first")
    handles.append(first)
    events.append("first acquired")
    try:
        events.append("second attempted")
        raise ValueError("cannot acquire second")
    except ValueError:
        first.close()
        events.append("first closed")
        raise

try:
    acquire_pair()
except ValueError:
    events.append("caller caught")
print(events, handles[0].closed)

observation_21 = {
    "01. 取得失败后的完整 events 顺序": "__未填写__",
    "02. 最后 handles[0].closed 的值": "__未填写__",
}

### 实验 21 即时检查

先运行本题代码，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 21)

### 实验 21 原因解释

调用方尚未得到完整的一对资源时，第一个资源由哪一层负责关闭？

## 实验 22：生成器提前关闭与资源退出

区分创建生成器、取得第一行、调用 `close()` 三个时刻。观察资源的关闭状态。

In [ ]:
from io import StringIO

events = []
streams = []

def iter_memory_lines():
    with StringIO("A\nB\n") as stream:
        streams.append(stream)
        events.append("entered")
        try:
            for line in stream:
                yield line.rstrip("\n")
        finally:
            events.append("leaving with")

source = iter_memory_lines()
before_consumption = events.copy()
first_line = next(source)
closed_after_first = streams[0].closed
source.close()
events_after_close = events.copy()
closed_after_close = streams[0].closed
print(before_consumption, first_line, closed_after_first)
print(events_after_close, closed_after_close)

observation_22 = {
    "01. 创建生成器后的 events": "__未填写__",
    "02. 第一次 next(source) 的结果": "__未填写__",
    "03. 第一行产出后 streams[0].closed 的值": "__未填写__",
    "04. close() 后 events 的顺序": "__未填写__",
    "05. close() 后 streams[0].closed 的值": "__未填写__",
}

### 实验 22 即时检查

先运行本题代码，再只检查当前实验。

In [ ]:
from chapter7_checks import check_experiment

check_experiment(globals(), 22)

### 实验 22 原因解释

为什么代码里已经写了 `with`，第一行产出后资源却仍被持有？哪个动作让生成器终于离开 `with`？

## 全部复查

22 道题各自通过即时检查后，再运行整章复查。它不代替逐题预测和原因解释。

In [ ]:
from chapter7_checks import run_checks

run_checks(globals())